In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import yfinance as yf
import datetime
import xgboost

from scipy.stats import norm
from scipy.integrate import quad
from scipy.optimize import minimize, least_squares, newton, differential_evolution, dual_annealing
from scipy.special import ndtr

from sklearn.model_selection import train_test_split
from sklearn.metrics import explained_variance_score, mean_absolute_percentage_error, mean_squared_error, mean_squared_log_error
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler
from sklearn.metrics import r2_score
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata
from scipy.optimize import brentq

from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from itertools import repeat
import random
import optuna
import json
import gc
optuna.logging.set_verbosity(optuna.logging.WARNING)

c:\Users\chril\torch_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Adapted ML code to output whole dataset (train + test data) instead of only test data

path = "C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\data_output_final\data_output_2002_top3_qual.dta"
data = pd.read_stata(path)
data = data.drop(["index"], axis = 1)


In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# Optuna hyperparameter optimisation

def _split_data(data, x_cols, y_col, ignore):
    X, Y = data[x_cols].copy(), data[[y_col]].copy()
    dates   = data["date"].sort_values().unique()
    cutoff  = dates[int(len(dates) * 0.7)]
    tmask   = data["date"] < cutoff
    tdates  = data.loc[tmask, "date"].sort_values().unique()
    vcutoff = tdates[int(len(tdates) * 0.9)]
    vmask   = tmask & (data["date"] >= vcutoff)
    trmask  = tmask & ~vmask
    non_cat = [c for c in X.columns if c not in ignore]
    scaler  = StandardScaler()
    X_tr = X[trmask].copy(); X_tr[non_cat] = scaler.fit_transform(X_tr[non_cat])
    X_v  = X[vmask].copy();  X_v[non_cat]  = scaler.transform(X_v[non_cat])
    return X_tr, Y[trmask], X_v, Y[vmask]


def _objective_xgb(trial, X_tr, Y_tr, X_v, Y_v):
    p = dict(
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3,  log=True),
        max_depth     = trial.suggest_int(  'max_depth',     3,    12),
        n_estimators  = trial.suggest_int(  'n_estimators',  50,   500),
        subsample     = trial.suggest_float('subsample',     0.5,  1.0),
        reg_lambda    = trial.suggest_float('reg_lambda',    0.01, 10.0, log=True),
        gamma         = trial.suggest_float('gamma',         0.0,  5.0),
        eval_metric='rmse', tree_method='hist', device='cuda',
    )
    m = xgboost.XGBRegressor(**p)
    m.fit(X_tr, Y_tr, eval_set=[(X_v, Y_v)], verbose=False)
    result = mean_squared_error(Y_v, m.predict(X_v))
    del m
    return result


def _objective_lbfgs(trial, Xtr_t, Ytr_t, Xv_t, Yv_t):
    n_layers     = trial.suggest_int(        'n_layers',     2,  4)
    n_units      = trial.suggest_categorical('n_units',      [32, 64, 128])
    max_iter     = trial.suggest_int(        'max_iter',     20, 50)
    history_size = trial.suggest_int(        'history_size', 5,  15)
    device       = Xtr_t.device
    try:
        layers = [nn.Linear(Xtr_t.shape[1], n_units), nn.GELU()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(n_units, n_units), nn.GELU()]
        layers.append(nn.Linear(n_units, 1))
        model = nn.Sequential(*layers).to(device)
        opt   = torch.optim.LBFGS(model.parameters(), max_iter=max_iter,
                                   history_size=history_size, line_search_fn='strong_wolfe')
        crit  = nn.MSELoss()

        best, patience, counter = float('inf'), 10, 0
        for epoch in range(30):
            model.train()
            xb, yb = Xtr_t, Ytr_t
            def closure():
                opt.zero_grad()
                loss = crit(model(xb), yb)
                if torch.isnan(loss) or torch.isinf(loss):
                    return loss
                loss.backward()
                return loss
            opt.step(closure)
            model.eval()
            with torch.no_grad():
                vl = crit(model(Xv_t), Yv_t).item()
            if np.isnan(vl) or np.isinf(vl):
                break  
            if vl < best:
                best, counter = vl, 0
            else:
                counter += 1
                if counter >= patience:
                    break
        del model, opt
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        return best
    except RuntimeError as e:
        if 'CUDA' in str(e):
            try:
                torch.cuda.synchronize()
            except RuntimeError:
                pass
            torch.cuda.empty_cache()
            return float('inf')
        raise


OPTUNA_N_TRIALS_XGB = 50
OPTUNA_N_TRIALS_ANN = 20

x_cols  = ['cp_flag_one', 'time_to_mat', 'rf', 'stock_price', 'realized_vol', 'moneyness']
ignore  = ['cp_flag_one']
targets = {'ivol': ('impl_volatility', False), 'price': ('price', True)}

optuna_best_params = {}
pruner  = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=20)
_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

set_seed(42)
for tname, (ycol, log_y) in targets.items():
    print(f"── {tname.upper()} ")
    X_tr, Y_tr, X_v, Y_v = _split_data(data, x_cols, ycol, ignore)

    _Xtr_t = torch.tensor(X_tr.values, dtype=torch.float32, device=_device)
    _Ytr_t = torch.tensor(Y_tr.values, dtype=torch.float32, device=_device)
    if log_y: _Ytr_t = torch.log(_Ytr_t.clamp(min=1e-6))
    _Xv_t  = torch.tensor(X_v.values,  dtype=torch.float32, device=_device)
    _Yv_t  = torch.tensor(Y_v.values,  dtype=torch.float32, device=_device)
    if log_y: _Yv_t  = torch.log(_Yv_t.clamp(min=1e-6))

    print(f"  XGB ({OPTUNA_N_TRIALS_XGB} trials)...")
    study = optuna.create_study(direction='minimize')
    study.optimize(lambda t: _objective_xgb(t, X_tr, Y_tr, X_v, Y_v),
                   n_trials=OPTUNA_N_TRIALS_XGB, show_progress_bar=True)
    optuna_best_params[f'xgb_{tname}'] = study.best_params
    print(f"    best val MSE: {study.best_value:.6f}  params: {study.best_params}")
    del study; gc.collect(); torch.cuda.synchronize(); torch.cuda.empty_cache()

    # ANN L-BFGS (pre-allocated GPU tensors passed in)
    print(f"  ANN-LBFGS ({OPTUNA_N_TRIALS_ANN} trials)...")
    study = optuna.create_study(direction='minimize')
    study.optimize(lambda t: _objective_lbfgs(t, _Xtr_t, _Ytr_t, _Xv_t, _Yv_t),
                   n_trials=OPTUNA_N_TRIALS_ANN, show_progress_bar=True, catch=(RuntimeError,))
    optuna_best_params[f'lbfgs_{tname}'] = study.best_params
    print(f"    best val MSE: {study.best_value:.6f}  params: {study.best_params}")
    del study; gc.collect(); torch.cuda.synchronize()

    # Free GPU tensors for this target before moving to the next
    del _Xtr_t, _Ytr_t, _Xv_t, _Yv_t
    gc.collect(); torch.cuda.synchronize(); torch.cuda.empty_cache()


with open('C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\ML_models\optuna_best_params_2002.json', 'w') as f:
    json.dump(optuna_best_params, f, indent=2)
print("Best parameters saved to /workspace/optuna_best_params.json")
print(json.dumps(optuna_best_params, indent=2))

── IVOL ──────────────────────────────────────────
  ANN-LBFGS (20 trials)...


Best trial: 6. Best value: 0.0019235:  75%|███████▌  | 15/20 [11:12<01:09, 13.83s/it]   

[W 2026-06-03 23:02:13,771] Trial 7 failed with parameters: {'n_layers': 4, 'n_units': 128, 'max_iter': 94, 'history_size': 8} because of the following error: RuntimeError('CUDA error: unknown error\nCUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.\nFor debugging consider passing CUDA_LAUNCH_BLOCKING=1\nCompile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.\n').
Traceback (most recent call last):
  File "C:\Users\chril\AppData\Local\Temp\ipykernel_28920\337086776.py", line 122, in _objective_lbfgs
    opt.step(closure)
  File "c:\Users\chril\torch_gpu\Lib\site-packages\torch\optim\optimizer.py", line 493, in wrapper
    out = func(*args, **kwargs)
          ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\chril\torch_gpu\Lib\site-packages\torch\utils\_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\chril\torch_gpu\Lib\site-packages\t

Best trial: 6. Best value: 0.0019235: 100%|██████████| 20/20 [11:13<00:00, 33.65s/it]


[W 2026-06-03 23:02:13,973] Trial 16 failed with parameters: {'n_layers': 3, 'n_units': 128, 'max_iter': 33, 'history_size': 30} because of the following error: RuntimeError('CUDA error: unknown error\nCUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.\nFor debugging consider passing CUDA_LAUNCH_BLOCKING=1\nCompile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.\n').
Traceback (most recent call last):
  File "C:\Users\chril\AppData\Local\Temp\ipykernel_28920\337086776.py", line 106, in _objective_lbfgs
    model = nn.Sequential(*layers).to(device)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\chril\torch_gpu\Lib\site-packages\torch\nn\modules\module.py", line 1343, in to
    return self._apply(convert)
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\chril\torch_gpu\Lib\site-packages\torch\nn\modules\module.py", line 903, in _apply
    module._apply(fn)
  File "c:\Users\chril\torch_gpu\Lib

RuntimeError: CUDA error: unknown error
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


Volatility Surface Estimation

In [ ]:
# Optuna: surface models (25 random days per trial)
import random as _random

OPTUNA_N_DAYS   = 25
OPTUNA_N_TRIALS_XGB_S = 50
OPTUNA_N_TRIALS_ANN_S = 20

x_cols_s = ['cp_flag_one', 'time_to_mat', 'rf', 'stock_price', 'realized_vol', 'moneyness']
ignore_s = ['cp_flag_one']
targets_s = {'ivol': ('impl_volatility', False), 'price': ('price', True)}


def _surface_sample(data, x_cols, y_col, ignore, n_days=25, seed=42):
    """Return list of (X_train, Y_train, X_val, Y_val) for n_days random days."""
    _random.seed(seed)
    all_dates = sorted(data['date'].unique())
    sampled   = _random.sample(all_dates, min(n_days, len(all_dates)))
    splits = []
    for d in sampled:
        grp = data[data['date'] == d]
        if len(grp) < 5:
            continue
        X = grp[x_cols].copy()
        Y = grp[[y_col]].copy()
        non_cat = [c for c in X.columns if c not in ignore]
        scaler = StandardScaler()
        X[non_cat] = scaler.fit_transform(X[non_cat])
        X_tr, X_v, Y_tr, Y_v = train_test_split(X, Y, train_size=0.7, shuffle=True, random_state=0)
        splits.append((X_tr, Y_tr, X_v, Y_v))
    return splits


def _obj_xgb_surface(trial, splits):
    p = dict(
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3,  log=True),
        max_depth     = trial.suggest_int(  'max_depth',     3,    12),
        n_estimators  = trial.suggest_int(  'n_estimators',  50,   500),
        subsample     = trial.suggest_float('subsample',     0.5,  1.0),
        reg_lambda    = trial.suggest_float('reg_lambda',    0.01, 10.0, log=True),
        gamma         = trial.suggest_float('gamma',         0.0,  5.0),
        eval_metric='rmse', tree_method='hist', device='cuda',
    )
    mses = []
    for X_tr, Y_tr, X_v, Y_v in splits:
        m = xgboost.XGBRegressor(**p)
        m.fit(X_tr, Y_tr, eval_set=[(X_v, Y_v)], verbose=False)
        mses.append(mean_squared_error(Y_v, m.predict(X_v)))
    return float(np.mean(mses))



def _obj_lbfgs_surface(trial, splits, log_y=False):
    n_layers     = trial.suggest_int(        'n_layers',     2,  4)
    n_units      = trial.suggest_categorical('n_units',      [32, 64, 128, 256])
    max_iter     = trial.suggest_int(        'max_iter',     5,  50)
    history_size = trial.suggest_int(        'history_size', 1,  10)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    mses = []
    for X_tr, Y_tr, X_v, Y_v in splits:
        layers = [nn.Linear(X_tr.shape[1], n_units), nn.GELU()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(n_units, n_units), nn.GELU()]
        layers.append(nn.Linear(n_units, 1))
        model = nn.Sequential(*layers).to(device)
        opt  = torch.optim.LBFGS(model.parameters(), max_iter=max_iter,
                                  history_size=history_size, line_search_fn='strong_wolfe')
        crit = nn.MSELoss()
        Xtr_t = torch.tensor(X_tr.values, dtype=torch.float32, device=device)
        Ytr_t = torch.tensor(Y_tr.values, dtype=torch.float32, device=device)
        if log_y: Ytr_t = torch.log(Ytr_t.clamp(min=1e-6))
        Xv_t  = torch.tensor(X_v.values,  dtype=torch.float32, device=device)
        Yv_t  = torch.tensor(Y_v.values,  dtype=torch.float32, device=device)
        if log_y: Yv_t = torch.log(Yv_t.clamp(min=1e-6))
        xb, yb = Xtr_t, Ytr_t
        for _ in range(5):
            def closure():
                opt.zero_grad(); crit(model(xb), yb).backward(); return crit(model(xb), yb)
            opt.step(closure)
        model.eval()
        with torch.no_grad():
            mses.append(crit(model(Xv_t), Yv_t).item())
    return float(np.mean(mses))


_bp_path = 'C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\ML_models\optuna_best_params_2003.json'
optuna_best_params_surface = json.load(open(_bp_path)) if __import__('os').path.exists(_bp_path) else {}

set_seed(42)
for tname, (ycol, log_y) in targets_s.items():
    print(f"\n── SURFACE {tname.upper()} ")
    splits = _surface_sample(data, x_cols_s, ycol, ignore_s, n_days=OPTUNA_N_DAYS)

    print(f"  XGB ({OPTUNA_N_TRIALS_XGB_S} trials, {len(splits)} days)...")
    study = optuna.create_study(direction='minimize')
    study.optimize(lambda t: _obj_xgb_surface(t, splits), n_trials=OPTUNA_N_TRIALS_XGB_S,
                   show_progress_bar=True)
    optuna_best_params_surface[f'xgb_surface_{tname}'] = study.best_params
    print(f"    best val MSE: {study.best_value:.6f}  params: {study.best_params}")
    del study; gc.collect(); torch.cuda.empty_cache()

    print(f"  ANN-LBFGS ({OPTUNA_N_TRIALS_ANN_S} trials)...")
    study = optuna.create_study(direction='minimize')
    study.optimize(lambda t: _obj_lbfgs_surface(t, splits, log_y=log_y),
                   n_trials=OPTUNA_N_TRIALS_ANN_S, show_progress_bar=True)
    optuna_best_params_surface[f'lbfgs_surface_{tname}'] = study.best_params
    print(f"    best val MSE: {study.best_value:.6f}  params: {study.best_params}")
    del study; gc.collect(); torch.cuda.empty_cache()

with open(_bp_path, 'w') as f:
    json.dump(optuna_best_params_surface, f, indent=2)
print("\nSurface params saved to optuna_best_params.json")


── SURFACE IVOL ──────────────────────────────────────
  XGB (50 trials, 25 days)...


Best trial: 45. Best value: 0.000475413: 100%|██████████| 50/50 [07:53<00:00,  9.47s/it]


    best val MSE: 0.000475  params: {'learning_rate': 0.24622092992266656, 'max_depth': 10, 'n_estimators': 390, 'subsample': 0.6878193379887605, 'reg_lambda': 1.7791727181856662, 'gamma': 0.0011979192678942645}
  ANN-Adam (20 trials)...


Best trial: 8. Best value: 0.000359098: 100%|██████████| 20/20 [01:49<00:00,  5.47s/it]


    best val MSE: 0.000359  params: {'n_layers': 3, 'n_units': 128, 'lr': 0.006046114496023196, 'dropout': 0.005099486334685422}
  ANN-LBFGS (20 trials)...


...

    best val MSE: 319.169478  params: {'learning_rate': 0.15826784847009204, 'max_depth': 5, 'n_estimators': 155, 'subsample': 0.66214172527501, 'reg_lambda': 9.995568297047843, 'gamma': 0.535686450392368}
  ANN-Adam (20 trials)...


Best trial: 16. Best value: 0.0226594: 100%|██████████| 20/20 [01:46<00:00,  5.31s/it]


    best val MSE: 0.022659  params: {'n_layers': 4, 'n_units': 256, 'lr': 0.005456985883210736, 'dropout': 0.08308946479689733}
  ANN-LBFGS (20 trials)...


Best trial: 14. Best value: 0.00246876: 100%|██████████| 20/20 [08:22<00:00, 25.10s/it]

    best val MSE: 0.002469  params: {'n_layers': 3, 'n_units': 32, 'max_iter': 45, 'history_size': 10}

Surface params saved to optuna_best_params.json


In [ ]:
# Optuna: variable models (first 60 trading days rolling window)
import gc
OPTUNA_N_DAYS_VAR   = 60
OPTUNA_N_TRIALS_VAR = 20

x_cols_v  = ['cp_flag_one', 'time_to_mat', 'rf', 'stock_price', 'realized_vol', 'moneyness']
ignore_v  = ['cp_flag_one']
targets_v = {'ivol': ('impl_volatility', False), 'price': ('price', True)}


def _obj_lbfgs_variable(trial, data_sub, x_cols, y_col, ignore, log_y=False):
    n_layers     = trial.suggest_int(        'n_layers',     2,  4)
    n_units      = trial.suggest_categorical('n_units',      [32, 64, 128, 256])
    max_iter     = trial.suggest_int(        'max_iter',     5,  50)
    history_size = trial.suggest_int(        'history_size', 1,  10)
    device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    layers = [nn.Linear(len(x_cols), n_units), nn.GELU()]
    for _ in range(n_layers - 1):
        layers += [nn.Linear(n_units, n_units), nn.GELU()]
    layers.append(nn.Linear(n_units, 1))
    model = nn.Sequential(*layers).to(device)
    opt  = torch.optim.LBFGS(model.parameters(), max_iter=max_iter,
                              history_size=history_size, line_search_fn='strong_wolfe')
    crit = nn.MSELoss()

    mses = []
    for d, grp in data_sub.groupby('date'):
        if len(grp) < 5: continue
        X = grp[x_cols].copy(); Y = grp[[y_col]].copy()
        non_cat = [c for c in X.columns if c not in ignore]
        scaler = StandardScaler()
        X[non_cat] = scaler.fit_transform(X[non_cat])
        X_tr, X_v, Y_tr, Y_v = train_test_split(X, Y, train_size=0.7, shuffle=True, random_state=0)
        Xtr_t = torch.tensor(X_tr.values, dtype=torch.float32, device=device)
        Ytr_t = torch.tensor(Y_tr.values, dtype=torch.float32, device=device)
        if log_y: Ytr_t = torch.log(Ytr_t.clamp(min=1e-6))
        Xv_t  = torch.tensor(X_v.values,  dtype=torch.float32, device=device)
        Yv_t  = torch.tensor(Y_v.values,  dtype=torch.float32, device=device)
        if log_y: Yv_t = torch.log(Yv_t.clamp(min=1e-6))
        xb, yb = Xtr_t, Ytr_t
        def closure(): opt.zero_grad(); l=crit(model(xb),yb); l.backward(); return l
        opt.step(closure)
        model.eval()
        with torch.no_grad():
            mses.append(crit(model(Xv_t), Yv_t).item())
    return float(np.mean(mses)) if mses else float('inf')


first_60_dates = sorted(data['date'].unique())[:OPTUNA_N_DAYS_VAR]
data_var_sub   = data[data['date'].isin(first_60_dates)]

_bp_path = 'C:\\Users\chril\Downloads\Master_Thesis_Materials\Data\Working\Option_Metrics\models_final_output\ML_models\optuna_best_params_2003.json'
optuna_best_params_var = json.load(open(_bp_path)) if __import__('os').path.exists(_bp_path) else {}

set_seed(42)
for tname, (ycol, log_y) in targets_v.items():
    print(f"\n── VARIABLE {tname.upper()} ")

    print(f"  ANN-LBFGS ({OPTUNA_N_TRIALS_VAR} trials)...")
    study = optuna.create_study(direction='minimize')
    study.optimize(lambda t: _obj_lbfgs_variable(t, data_var_sub, x_cols_v, ycol, ignore_v, log_y=log_y),
                   n_trials=OPTUNA_N_TRIALS_VAR, show_progress_bar=True)
    optuna_best_params_var[f'lbfgs_variable_{tname}'] = study.best_params
    print(f"    best val MSE: {study.best_value:.6f}  params: {study.best_params}")
    del study; gc.collect(); torch.cuda.empty_cache()

with open(_bp_path, 'w') as f:
    json.dump(optuna_best_params_var, f, indent=2)
print("\nVariable params saved to optuna_best_params.json")



── VARIABLE IVOL ─────────────────────────────────────
  ANN-Adam (20 trials, 60 days)...


Best trial: 13. Best value: 0.00774918: 100%|██████████| 20/20 [00:08<00:00,  2.35it/s]


    best val MSE: 0.007749  params: {'n_layers': 2, 'n_units': 64, 'lr': 0.009641184960214228}
  ANN-LBFGS (20 trials)...


Best trial: 10. Best value: 0.000181292:  85%|████████▌ | 17/20 [03:00<00:37, 12.49s/it]

[W 2026-06-01 23:34:48,079] Trial 16 failed with parameters: {'n_layers': 3, 'n_units': 32, 'max_iter': 42, 'history_size': 8} because of the following error: The value nan is not acceptable.
[W 2026-06-01 23:34:48,079] Trial 16 failed with value nan.


...

[W 2026-06-01 23:38:25,327] Trial 15 failed with parameters: {'n_layers': 4, 'n_units': 32, 'max_iter': 50, 'history_size': 10} because of the following error: The value nan is not acceptable.
[W 2026-06-01 23:38:25,327] Trial 15 failed with value nan.


Best trial: 14. Best value: 0.0023102:  90%|█████████ | 18/20 [03:08<00:25, 12.59s/it]

[W 2026-06-01 23:38:41,252] Trial 17 failed with parameters: {'n_layers': 4, 'n_units': 32, 'max_iter': 50, 'history_size': 8} because of the following error: The value nan is not acceptable.
[W 2026-06-01 23:38:41,252] Trial 17 failed with value nan.


Best trial: 14. Best value: 0.0023102: 100%|██████████| 20/20 [03:39<00:00, 11.00s/it]

    best val MSE: 0.002310  params: {'n_layers': 4, 'n_units': 32, 'max_iter': 50, 'history_size': 10}

Variable params saved to optuna_best_params.json
